In [6]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("Instacart-Retail").getOrCreate()

BRONZE = "/home/jovyan/work/data/bronze"
order_products_prior = spark.read.parquet(f"{BRONZE}/order_products_prior")
products = spark.read.parquet(f"{BRONZE}/products")
departments = spark.read.parquet(f"{BRONZE}/departments")
orders = spark.read.parquet(f"{BRONZE}/orders")


In [2]:
# ============================================
# Q1: Top 10 most ordered products
# (Business Q: SKU ไหนขายดีสุด)
# ============================================
top_products = (order_products_prior
    .groupBy("product_id")
    .agg(F.count("*").alias("order_count"))
    .join(products, on="product_id")
    .orderBy(F.desc("order_count"))
    .limit(10))

top_products.show(truncate=False)
# คาดว่าจะเห็น: Banana, Bag of Organic Bananas, Organic Strawberries...

+----------+-----------+----------------------+--------+-------------+
|product_id|order_count|product_name          |aisle_id|department_id|
+----------+-----------+----------------------+--------+-------------+
|24852     |472565     |Banana                |24      |4            |
|13176     |379450     |Bag of Organic Bananas|24      |4            |
|21137     |264683     |Organic Strawberries  |24      |4            |
|21903     |241921     |Organic Baby Spinach  |123     |4            |
|47209     |213584     |Organic Hass Avocado  |24      |4            |
|47766     |176815     |Organic Avocado       |24      |4            |
|47626     |152657     |Large Lemon           |24      |4            |
|16797     |142951     |Strawberries          |24      |4            |
|26209     |140627     |Limes                 |24      |4            |
|27845     |137905     |Organic Whole Milk    |84      |16           |
+----------+-----------+----------------------+--------+-------------+



In [3]:
# ============================================
# Q2: Reorder rate per department
# (Business Q: department ไหนลูกค้าซื้อซ้ำบ่อยสุด — สำคัญใน FMCG)
# ============================================
reorder_by_dept = (order_products_prior
    .join(products, on="product_id")
    .join(departments, on="department_id")
    .groupBy("department")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("reordered").alias("reorder_count"),
        F.round(F.avg("reordered"), 3).alias("reorder_rate")
    )
    .orderBy(F.desc("reorder_rate")))

reorder_by_dept.show(25, truncate=False)
# Insight: dairy eggs, beverages, produce → reorder rate สูง (consumable)
#          personal care, household → ต่ำ (durable)

+---------------+------------+-------------+------------+
|department     |total_orders|reorder_count|reorder_rate|
+---------------+------------+-------------+------------+
|dairy eggs     |5414016     |3627221      |0.67        |
|beverages      |2690129     |1757892      |0.653       |
|produce        |9479291     |6160710      |0.65        |
|bakery         |1176787     |739188       |0.628       |
|deli           |1051249     |638864       |0.608       |
|pets           |97724       |58760        |0.601       |
|babies         |423802      |245369       |0.579       |
|bulk           |34573       |19950        |0.577       |
|snacks         |2887550     |1657973      |0.574       |
|alcohol        |153696      |87595        |0.57        |
|meat seafood   |708931      |402442       |0.568       |
|breakfast      |709569      |398013       |0.561       |
|frozen         |2236432     |1211890      |0.542       |
|dry goods pasta|866627      |399581       |0.461       |
|canned goods 

In [7]:
# ============================================
# Q3: Order pattern by day of week + hour
# (Business Q: peak hour ไหน — สำหรับ workforce planning)
# ============================================
dow_map = {0:"Sun", 1:"Mon", 2:"Tue", 3:"Wed", 4:"Thu", 5:"Fri", 6:"Sat"}
dow_expr = F.create_map([F.lit(x) for kv in dow_map.items() for x in kv])

orders_pattern = (orders
    .withColumn("dow_name", dow_expr[F.col("order_dow")])
    .groupBy("dow_name", "order_hour_of_day")
    .agg(F.count("*").alias("order_count"))
    .orderBy("dow_name", "order_hour_of_day"))

orders_pattern.show(20)

+--------+-----------------+-----------+
|dow_name|order_hour_of_day|order_count|
+--------+-----------------+-----------+
|     Fri|                0|       3189|
|     Fri|                1|       1672|
|     Fri|                2|       1016|
|     Fri|                3|        841|
|     Fri|                4|        910|
|     Fri|                5|       1574|
|     Fri|                6|       4866|
|     Fri|                7|      13434|
|     Fri|                8|      24015|
|     Fri|                9|      34232|
|     Fri|               10|      38313|
|     Fri|               11|      37915|
|     Fri|               12|      35714|
|     Fri|               13|      36296|
|     Fri|               14|      37407|
|     Fri|               15|      37508|
|     Fri|               16|      35860|
|     Fri|               17|      29955|
|     Fri|               18|      24310|
|     Fri|               19|      18741|
+--------+-----------------+-----------+
only showing top